In [0]:
%run ../00_Setup/01_Config

In [0]:
%run ../Utils/Common_Utils

In [0]:
PIPELINE_NAME = "Gold Product Ranking Analysis"
SOURCE_TABLE = GOLD_FACT_SALES
TARGET_TABLE = GOLD_PRODUCT_RANKING
RUN_ID = generate_run_id()
START_TIME = datetime.now()

In [0]:
print("GOLD PRODUCT RANKING PIPELINE")

print(f"Pipeline : {PIPELINE_NAME}")
print(f"Run ID : {RUN_ID}")
print(f"Target : {TARGET_TABLE}")

GOLD PRODUCT RANKING PIPELINE
Pipeline : Gold Product Ranking Analysis
Run ID : d4c1ef89-0bf5-4539-b47e-7dd546a79e9e
Target : retailmart.gold.product_ranking


In [0]:
fact_sales_df = spark.table(SOURCE_TABLE)
display(fact_sales_df.limit(10))

order_id,order_item_id,order_status,order_purchase_timestamp,order_year,order_month,revenue_month,order_delivered_customer_date,delivery_duration_days,customer_id,customer_city,customer_state,product_id,product_category_name,price,freight_value,total_item_value,payment_type,payment_installments,total_payment_value
ORD_0000001,1,delivered,2023-06-15T14:30:00.000Z,2023,6,2023-06,2023-06-19T14:30:00.000Z,4,CUST_006571,Belo Horizonte,MG,PROD_001950,books,1754.7,20.17,1774.87,multiple,12,2807.85
ORD_0000002,1,cancelled,2021-06-12T11:02:00.000Z,2021,6,2021-06,null,null,CUST_006956,Aracaju,SE,PROD_000989,food,2105.2,45.48,2150.68,voucher,1,805.73
ORD_0000003,1,delivered,2022-03-31T19:38:00.000Z,2022,3,2022-03,2022-04-14T19:38:00.000Z,14,CUST_008373,Joao Pessoa,PB,PROD_000254,furniture,1627.94,78.29,1706.23,credit_card,6,508.88
ORD_0000004,1,delivered,2021-09-09T22:29:00.000Z,2021,9,2021-09,2021-09-23T22:29:00.000Z,14,CUST_007986,Belo Horizonte,MG,PROD_001327,music,65.22,31.08,96.3,boleto,3,1624.71
ORD_0000004,2,delivered,2021-09-09T22:29:00.000Z,2021,9,2021-09,2021-09-23T22:29:00.000Z,14,CUST_007986,Belo Horizonte,MG,PROD_002714,computers,946.39,44.43,990.82,boleto,3,1624.71
ORD_0000004,3,delivered,2021-09-09T22:29:00.000Z,2021,9,2021-09,2021-09-23T22:29:00.000Z,14,CUST_007986,Belo Horizonte,MG,PROD_001507,home_appliances,157.96,55.93,213.89,boleto,3,1624.71
ORD_0000005,1,shipped,2022-08-27T07:46:00.000Z,2022,8,2022-08,null,null,CUST_002810,Manaus,AM,PROD_001872,fashion,1681.83,14.0,1695.83,credit_card,3,1592.91
ORD_0000006,1,invoiced,2023-05-26T16:27:00.000Z,2023,5,2023-05,null,null,CUST_007867,Recife,PE,PROD_001518,garden,856.74,64.83,921.57,credit_card,1,495.97
ORD_0000007,1,processing,2021-09-21T15:50:00.000Z,2021,9,2021-09,null,null,CUST_004827,Campo Grande,MS,PROD_001504,music,373.8,66.88,440.68,credit_card,2,1738.92
ORD_0000008,1,delivered,2021-01-19T14:24:00.000Z,2021,1,2021-01,2021-01-28T14:24:00.000Z,9,CUST_005137,Porto Velho,RO,PROD_001133,health,899.98,23.95,923.93,voucher,1,1107.69


In [0]:
print(f"Total Records : {fact_sales_df.count()}")
fact_sales_df.printSchema()

Total Records : 86328
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_month: string (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- delivery_duration_days: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- total_item_value: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- total_payment_value: double (nullable = true)



In [0]:
# Top category by revenue
spark.sql(f"""
SELECT
    product_category_name,
    ROUND(SUM(total_item_value),2) AS total_revenue
FROM {SOURCE_TABLE}
GROUP BY product_category_name
ORDER BY total_revenue DESC
""").show(truncate=False)

+---------------------+-------------+
|product_category_name|total_revenue|
+---------------------+-------------+
|garden               |8670791.66   |
|furniture            |8398532.0    |
|toys                 |8107064.65   |
|books                |8106409.08   |
|health               |7817553.88   |
|automotive           |7735888.78   |
|electronics          |7497453.46   |
|music                |7275179.86   |
|fashion              |7162874.72   |
|beauty               |7078046.5    |
|computers            |7021460.45   |
|food                 |6943258.77   |
|home_appliances      |6903585.75   |
|office               |6846870.26   |
|sports               |6726251.72   |
+---------------------+-------------+



In [0]:
# Top selling products
spark.sql(f"""
SELECT
    product_category_name,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(total_item_value),2) AS total_revenue
FROM {SOURCE_TABLE}
GROUP BY product_category_name

ORDER BY total_orders DESC
""").show(truncate=False)

+---------------------+------------+-------------+
|product_category_name|total_orders|total_revenue|
+---------------------+------------+-------------+
|garden               |6342        |8670791.66   |
|furniture            |6094        |8398532.0    |
|books                |6046        |8106409.08   |
|toys                 |5902        |8107064.65   |
|health               |5769        |7817553.88   |
|automotive           |5609        |7735888.78   |
|electronics          |5552        |7497453.46   |
|music                |5315        |7275179.86   |
|fashion              |5249        |7162874.72   |
|beauty               |5236        |7078046.5    |
|food                 |5077        |6943258.77   |
|computers            |5073        |7021460.45   |
|home_appliances      |5058        |6903585.75   |
|office               |5054        |6846870.26   |
|sports               |5018        |6726251.72   |
+---------------------+------------+-------------+



In [0]:
# Top products
spark.sql(f"""
SELECT
    product_id,
    product_category_name,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(total_item_value),2) AS total_revenue
FROM {SOURCE_TABLE}
GROUP BY product_id, product_category_name
ORDER BY total_revenue DESC
LIMIT 10
""").show(truncate=False)

+-----------+---------------------+------------+-------------+
|product_id |product_category_name|total_orders|total_revenue|
+-----------+---------------------+------------+-------------+
|PROD_000005|toys                 |51          |68731.04     |
|PROD_001490|computers            |44          |64849.85     |
|PROD_001159|health               |47          |63777.84     |
|PROD_001488|electronics          |47          |62075.97     |
|PROD_000593|home_appliances      |42          |61992.87     |
|PROD_002467|furniture            |42          |61792.02     |
|PROD_001171|computers            |40          |61037.87     |
|PROD_000231|home_appliances      |41          |60627.24     |
|PROD_000480|health               |50          |60503.48     |
|PROD_001829|toys                 |43          |60269.29     |
+-----------+---------------------+------------+-------------+



In [0]:
spark.sql(f"""

CREATE OR REPLACE TABLE {TARGET_TABLE} AS

WITH product_summary AS
(

SELECT
    product_id,
    product_category_name,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(total_item_value),2) AS total_revenue,
    ROUND(AVG(total_item_value),2) AS average_item_value

FROM {SOURCE_TABLE}

GROUP BY
    product_id,
    product_category_name
),

product_ranking AS
(

SELECT
    product_id,
    product_category_name,
    total_orders,
    total_revenue,
    average_item_value,

    ROW_NUMBER() OVER(
        ORDER BY total_revenue DESC
    ) AS row_number,

    RANK() OVER(
        ORDER BY total_revenue DESC
    ) AS revenue_rank,

    DENSE_RANK() OVER(
        ORDER BY total_revenue DESC
    ) AS dense_revenue_rank,

    RANK() OVER(
        PARTITION BY product_category_name
        ORDER BY total_revenue DESC
    ) AS category_rank

FROM product_summary

)

SELECT *
FROM product_ranking
ORDER BY revenue_rank,product_id
""")


DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
product_ranking_df = spark.table(TARGET_TABLE)
display(product_ranking_df.limit(10))

product_id,product_category_name,total_orders,total_revenue,average_item_value,row_number,revenue_rank,dense_revenue_rank,category_rank
PROD_000005,toys,51,68731.04,1347.67,1,1,1,1
PROD_001490,computers,44,64849.85,1473.86,2,2,2,1
PROD_001159,health,47,63777.84,1356.98,3,3,3,1
PROD_001488,electronics,47,62075.97,1320.77,4,4,4,1
PROD_000593,home_appliances,42,61992.87,1476.02,5,5,5,1
PROD_002467,furniture,42,61792.02,1471.24,6,6,6,1
PROD_001171,computers,40,61037.87,1525.95,7,7,7,2
PROD_000231,home_appliances,41,60627.24,1478.71,8,8,8,2
PROD_000480,health,50,60503.48,1210.07,9,9,9,2
PROD_001829,toys,43,60269.29,1401.61,10,10,10,2


In [0]:
# Validation1
rows_written = product_ranking_df.count()
print(f"Rows Written : {rows_written}")

Rows Written : 3000


In [0]:
# Validation2
product_ranking_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- average_item_value: double (nullable = true)
 |-- row_number: integer (nullable = true)
 |-- revenue_rank: integer (nullable = true)
 |-- dense_revenue_rank: integer (nullable = true)
 |-- category_rank: integer (nullable = true)



In [0]:
# Validation3
display(product_ranking_df.describe())

summary,product_id,product_category_name,total_orders,total_revenue,average_item_value,row_number,revenue_rank,dense_revenue_rank,category_rank
count,3000,3000,3000,3000,3000,3000,3000,3000,3000
mean,null,null,28.77166666666667,37430.4071800001,1301.1248033333331,1500.5,1500.5,1500.5,101.07133333333333
stddev,null,null,5.367538297768819,7928.158109796272,138.04987311131435,866.1697293256098,866.1697293256098,866.1697293256098,58.75734609501121
min,PROD_000001,automotive,11,11030.08,819.77,1,1,1,1
max,PROD_003000,toys,51,68731.04,1833.4,3000,3000,3000,233


In [0]:
assert rows_written > 0

In [0]:
bronze_load_report(

    pipeline_name=PIPELINE_NAME,
    run_id=RUN_ID,
    source=SOURCE_TABLE,
    target=TARGET_TABLE,
    rows_read=product_ranking_df.count(),
    rows_written=rows_written,
    duplicate_count=0,
    start_time=START_TIME,
    status="SUCCESS"

)

LOAD REPORT
Pipeline        : Gold Product Ranking Analysis
Run ID          : d4c1ef89-0bf5-4539-b47e-7dd546a79e9e
Source          : retailmart.gold.fact_sales
Target          : retailmart.gold.product_ranking
Rows Read       : 3000
Rows Written    : 3000
Duplicate Rows  : 0
Start Time      : 2026-07-19 10:29:45.292108
End Time        : 2026-07-19 10:30:04.277673
Duration (sec)  : 18.99
Status          : SUCCESS


# Engineering Observations
• Aggregated product-level sales metrics from the Gold Fact Sales table.
• Applied SQL Window Functions (ROW_NUMBER, RANK, DENSE_RANK) to rank products based on revenue.
• Calculated category-wise product rankings using PARTITION BY.
• Built a Gold-layer analytical table to identify top-performing products.
• Produced a business-ready dataset for dashboards and reporting.